<a href="https://colab.research.google.com/github/PrantoMondol11/2.1/blob/main/Knowledage%20Distilation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

In [4]:
transform=transforms.ToTensor()
train_data=datasets.MNIST(root='./data',train=True,download=True,transform=transform)
train_loader=DataLoader(train_data,batch_size=64,shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 475kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.48MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.88MB/s]


In [5]:
teacher=nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28,256),
    nn.ReLU(),
    nn.Linear(256,10)
)

In [6]:
student=nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28,64),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(64,10)
)

In [7]:
optimizer_t=optim.Adam(teacher.parameters(),lr=0.001)
loss_fn=nn.CrossEntropyLoss()

for epoch in range(2):
  for x,y in train_loader:
    pred=teacher(x)
    loss=loss_fn(pred,y)

    optimizer_t.zero_grad()
    loss.backward()
    optimizer_t.step()

In [8]:

optimizer=optim.Adam(student.parameters(),lr=0.001)
temperature=3
alpha=.5
ce_loss=nn.CrossEntropyLoss()
kl_loss=nn.KLDivLoss(reduce='batchmean')

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


In [9]:
for epoch in range(2):
    for x, y in train_loader:
       with torch.no_grad():
            teacher_logits = teacher(x)
       student_logits = student(x)
       loss_hard = ce_loss(student_logits, y)
       soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
       soft_student = F.log_softmax(student_logits / temperature, dim=1)
       loss_soft = kl_loss(soft_student, soft_teacher)
       loss = alpha * loss_hard + (1 - alpha) * loss_soft
       optimizer.zero_grad()
       loss.backward()
       optimizer.step()


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:558: UserWarning: reduction: 'mean' divides the total loss by both the batch size and the support size.'batchmean' divides only by the batch size, and aligns with the KL div math definition.'mean' will be changed to behave the same as 'batchmean' in the next major release.
  return F.kl_div(


In [10]:
correct = 0
total = 0

with torch.no_grad():
    for x, y in train_loader:
        outputs = student(x)
        _, predicted = torch.max(outputs, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

print("Accuracy:", correct / total)

Accuracy: 0.9059666666666667


In [11]:
from sklearn.metrics import accuracy_score
y_true = []
y_pred = []

with torch.no_grad():
    for x, y in train_loader:
        outputs = student(x)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(y.numpy())
        y_pred.extend(predicted.numpy())
    acc = accuracy_score(y_true, y_pred)
    print("Accuracy:", acc)

Accuracy: 0.90745


In [12]:
import torch
import os

# save model
torch.save(student.state_dict(), "student1.pth")

# check size
size = os.path.getsize("student1.pth") / 1024**2  # MB
print(f"Model size: {size:.2f} MB")

Model size: 0.20 MB


In [13]:
from ast import mod
import torch.nn.utils.prune as prune

for module in student.modules():
  if isinstance(module,torch.nn.Linear):
    prune.l1_unstructured(module,name='weight',amount=0.3)

for name,module in student.named_modules():
  if isinstance(module,torch.nn.Linear):
    print(name,'Zeros:',float(torch.sum(module.weight==0))/module.weight.nelement())


1 Zeros: 0.30000398596938777
4 Zeros: 0.3


In [14]:
for module in student.modules():
  if isinstance(module,torch.nn.Linear):
    prune.remove(module,'weight')

In [15]:
acc = accuracy_score(y_true, y_pred)
print("After pruning accuracy:", acc)

After pruning accuracy: 0.90745


In [16]:
from sklearn.metrics import accuracy_score
y_true = []
y_pred = []

with torch.no_grad():
    for x, y in train_loader:
        outputs = student(x)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(y.numpy())
        y_pred.extend(predicted.numpy())
    acc = accuracy_score(y_true, y_pred)
    print("Accuracy:", acc)

Accuracy: 0.8966333333333333


In [17]:
import torch
import os

# save model
torch.save(student.state_dict(), "student.pth")

# check size
size = os.path.getsize("student.pth") / 1024**2  # MB
print(f"Model size: {size:.2f} MB")

Model size: 0.20 MB


In [18]:
student.train()

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=64, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=64, out_features=10, bias=True)
)

In [19]:
import torch

quantized_model = torch.quantization.quantize_dynamic(
    student,
    {torch.nn.Linear},  # layers to quantize
    dtype=torch.qint8
)

/tmp/ipykernel_2334/258694889.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [20]:
import os

torch.save(student.state_dict(), "student.pth")
torch.save(quantized_model.state_dict(), "student_quant.pth")

print("Original size:", os.path.getsize("student.pth"))
print("Quantized size:", os.path.getsize("student_quant.pth"))

Original size: 205917
Quantized size: 55009


In [21]:
y_true = []
y_pred = []

with torch.no_grad():
    for x, y in train_loader:
        outputs = quantized_model(x)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(y.numpy())
        y_pred.extend(predicted.numpy())

from sklearn.metrics import accuracy_score
print("Quantized Accuracy:", accuracy_score(y_true, y_pred))

Quantized Accuracy: 0.9415666666666667


In [22]:
import torch
import os

# save model
torch.save(student.state_dict(), "student.pth")

# check size
size = os.path.getsize("student.pth") / 1024**2  # MB
print(f"Model size: {size:.2f} MB")

Model size: 0.20 MB


In [23]:
import torch

# save original student
torch.save(student.state_dict(), "student.pth")

# save quantized model
torch.save(quantized_model.state_dict(), "student_quant.pth")

In [24]:
import os

size_original = os.path.getsize("student.pth") / 1024  # KB
size_quant = os.path.getsize("student_quant.pth") / 1024  # KB

print("Original size: {:.2f} KB".format(size_original))
print("Quantized size: {:.2f} KB".format(size_quant))

Original size: 201.09 KB
Quantized size: 53.72 KB


In [25]:
import torch
import torch.nn.functional as F

def mc_dropout_predict(model, x, n_samples=20):
    preds = []

    for _ in range(n_samples):
        output = model(x)  # dropout active
        prob = F.softmax(output, dim=1)
        preds.append(prob)

    preds = torch.stack(preds)   # [samples, batch, classes]

    mean = preds.mean(dim=0)     # average prediction
    std = preds.std(dim=0)       # uncertainty

    return mean, std

In [26]:
x, y = next(iter(train_loader))

mean, std = mc_dropout_predict(student, x)

pred = torch.argmax(mean, dim=1)

print("Prediction:", pred[0].item())
print("Uncertainty:", std[0])

Prediction: 4
Uncertainty: tensor([0.0014, 0.0003, 0.0038, 0.0055, 0.0581, 0.0038, 0.0012, 0.0291, 0.0027,
        0.0311], grad_fn=<SelectBackward0>)
